In [1]:
import os
import glob
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime
import chardet

# KT20 cond and mag

In [2]:
def find_missing_list_indices(dataframe, column_name, value_list):
    """
    Align `dataframe` to the full `value_list` of expected sample ids.
    A value_list entry with no matching row in `dataframe` at all means
    that sample was never logged in this sheet — treated as case 1
    (blank row: sample id restored, every other cell left as real NaN).
    Rows already present with 'missing' in other columns (case 2, set by
    the calling _proc function) are left untouched.
    """
    df = dataframe.copy()
    df = df[df[column_name] != 'del'].reset_index(drop=True)

    def last_four_key(x):
        if pd.isna(x):
            return None
        try:
            return int(str(x)[-4:])
        except ValueError:
            return None

    df['_key'] = df[column_name].apply(last_four_key)

    value_keys = [int(str(x)[-4:]) for x in value_list]
    if pd.Series(value_keys).duplicated().any():
        dupes = pd.Series(value_keys)[pd.Series(value_keys).duplicated()].unique().tolist()
        raise ValueError(f"value_list has non-unique last-four-digit keys: {dupes}")

    df = df[df['_key'].notna()].drop_duplicates(subset='_key')
    df = df.set_index('_key').reindex(value_keys)
    df = df.drop(columns=['_key'], errors='ignore')

    # A row that's entirely NaN (including column_name) is one reindex just
    # introduced — i.e. this sample id had no row in the sheet at all.
    never_logged_mask = df.isna().all(axis=1)
    if never_logged_mask.any():
        ids = [v for v, m in zip(value_list, never_logged_mask) if m]
        print(f"Note: {len(ids)} sample id(s) not present in this sheet at all: {ids}")

    df.loc[never_logged_mask, column_name] = [
        str(v) for v, m in zip(value_list, never_logged_mask) if m
    ]

    return df.reset_index(drop=True)

In [3]:
HEADER_ROWS = 2   # rows to skip before numeric data starts
SIGMA = 2         # outlier threshold in std devs


def _load_measurement_stats(csv_path, header_rows=HEADER_ROWS, sigma=SIGMA):
    """Read one measurement CSV and return outlier-filtered summary stats, or None if unusable."""
    df = pd.read_csv(csv_path)

    min_marker_rows = df.index[df['Date'] == 'Min'].to_list()
    if len(min_marker_rows) != 1:
        return None
    end_row = min_marker_rows[0]

    raw_values = np.array(df.iloc[header_rows:end_row, 1]).astype('float32')
    raw_values = np.reshape(raw_values, (end_row - header_rows, 1))

    mean = raw_values.astype('float64').mean()
    std = raw_values.astype('float64').std()

    # masked_outside(x, v1, v2) masks x < v1 or x > v2 — lower bound first
    masked = np.ma.masked_outside(raw_values, mean - sigma * std, mean + sigma * std)

    if masked.compressed().size == 0:   # all values flagged as outliers
        return None

    return {
        'mean': np.mean(masked),
        'std': np.std(masked),
        'median': np.median(masked),
        'max': np.max(masked),
        'min': np.min(masked),
    }


def _read_meta_data(csv_path):
    """Read the kappa_sigma meta CSV, trying comma then semicolon delimiter."""
    for sep in (',', ';'):
        meta_data = pd.read_csv(csv_path, sep=sep)
        if 'Record_Id' in meta_data.columns:
            return meta_data
    raise ValueError(f"Could not find 'Record_Id' column in {csv_path} with ',' or ';' delimiter")


def cond_sus_proc(drillhole, prop, header_rows=HEADER_ROWS, sigma=SIGMA):
    base_dir = Path(drillhole)
    cond_sus_dir = base_dir / 'cond_sus'

    data_inventory = pd.read_excel(base_dir / f'{drillhole}_sample_report.xlsx', sheet_name=None)
    df_record = data_inventory.get(prop)
    df_report = data_inventory.get('sample_report')
    sample_id_report = df_report['sample_id'].to_list()

    df_record = df_record.reset_index(drop=True)

    dated_folders = []
    for name in os.listdir(cond_sus_dir):
        try:
            dated_folders.append((datetime.strptime(name, "%d-%m-%y"), name))
        except ValueError:
            continue
    sorted_folder_names = [name for _, name in sorted(dated_folders)]

    results = []
    stat_cols = ['mean', 'std', 'median', 'max', 'min']

    for folder_name in sorted_folder_names:
        instrument_dir = cond_sus_dir / folder_name / 'kt20_0385'
        meta_csv = instrument_dir / 'kappa_sigma_0385.csv'
        if not meta_csv.exists():
            print(f"Skipping {folder_name}: no meta CSV found")
            continue

        meta_data = _read_meta_data(meta_csv)
        df_record_sub = df_record[df_record['measurement_id'].isin(meta_data['Record_Id'])]

        for orig_idx, row in df_record_sub.iterrows():
            record_dir = instrument_dir / str(int(row['measurement_id']))
            csv_files = glob.glob(str(record_dir / '*.csv'))
            if not csv_files:
                continue

            stats = _load_measurement_stats(csv_files[0], header_rows=header_rows, sigma=sigma)
            if stats is None:
                continue

            results.append({'orig_index': orig_idx, **stats})

    df_proc = pd.DataFrame(results).set_index('orig_index') if results else pd.DataFrame(columns=stat_cols)
    df_proc = df_proc.reindex(df_record.index)

    # sample id is always filled, whether or not a measurement was found —
    # case 1 rows (measurement_id never recorded) still show their sample id.
    df_proc.insert(0, 'sample id', df_record['sample_id'].astype(str).values)

    not_recorded = df_record['measurement_id'].isna().values
    no_data_found = df_proc[stat_cols].isna().all(axis=1).values

    # Case 2: measurement_id WAS recorded, but no usable data was found for it
    data_missing_mask = no_data_found & ~not_recorded
    df_proc.loc[data_missing_mask, :] = 'missing'
    df_proc.loc[data_missing_mask, 'sample id'] = df_record.loc[data_missing_mask, 'sample_id'].astype(str).values

    cal_mask = df_proc['sample id'].str.contains('cal', na=False)
    df_cal = df_proc[cal_mask]
    df_final = df_proc[~cal_mask]
    df_final.index = range(len(df_final))

    # Cross-check against the full sample_report: a sample id entirely absent
    # from this sheet is case 1 (blank), same treatment as an explicit NaN.
    df_final = find_missing_list_indices(df_final, 'sample id', sample_id_report)

    return df_final, df_cal

In [4]:
d1, d2 = cond_sus_proc('TC-2531-001','sus')

c:\Users\Prithwijit\anaconda3\envs\dataanalysis140323\lib\site-packages\numpy\core\fromnumeric.py:758: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  a.partition(kth, axis=axis, kind=kind, order=order)
c:\Users\Prithwijit\anaconda3\envs\dataanalysis140323\lib\site-packages\numpy\core\fromnumeric.py:758: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  a.partition(kth, axis=axis, kind=kind, order=order)
c:\Users\Prithwijit\anaconda3\envs\dataanalysis140323\lib\site-packages\numpy\core\fromnumeric.py:758: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  a.partition(kth, axis=axis, kind=kind, order=order)
c:\Users\Prithwijit\anaconda3\envs\dataanalysis140323\lib\site-packages\numpy\core\fromnumeric.py:758: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  a.partition(kth, axis=axis, kind=kind, order=order)
c:\Users\Prithwijit\anaconda3\envs\dataanalysis140323\lib\site-p

Note: 17 sample id(s) not present in this sheet at all: [109807, 109808, 109809, 109810, 109811, 109812, 109813, 109814, 109815, 109816, 109817, 109818, 109819, 109820, 109821, 109822, 109823]


c:\Users\Prithwijit\anaconda3\envs\dataanalysis140323\lib\site-packages\numpy\core\fromnumeric.py:758: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  a.partition(kth, axis=axis, kind=kind, order=order)
c:\Users\Prithwijit\anaconda3\envs\dataanalysis140323\lib\site-packages\numpy\core\fromnumeric.py:758: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  a.partition(kth, axis=axis, kind=kind, order=order)
c:\Users\Prithwijit\anaconda3\envs\dataanalysis140323\lib\site-packages\numpy\core\fromnumeric.py:758: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  a.partition(kth, axis=axis, kind=kind, order=order)
c:\Users\Prithwijit\anaconda3\envs\dataanalysis140323\lib\site-packages\numpy\core\fromnumeric.py:758: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  a.partition(kth, axis=axis, kind=kind, order=order)
c:\Users\Prithwijit\anaconda3\envs\dataanalysis140323\lib\site-p

In [5]:
d1.to_csv('check_sus.csv')

# Density

In [6]:
def density(drillhole):
    # Load every sheet in the sample report workbook at once (sheet_name=None
    # returns a dict of {sheet_name: DataFrame})
    data_inventory = pd.read_excel(f'{drillhole}/{drillhole}_sample_report.xlsx', sheet_name=None)

    # Master list of expected sample IDs — used later to flag any that are
    # missing from the density measurements
    df_report = data_inventory.get('sample_report')
    sample_id_report = df_report['sample_id']

    # Pull the density-specific sheet; fail loudly with a clear message if the
    # sheet name doesn't exist, rather than a confusing NoneType error later
    df_den = data_inventory.get('den')
    if df_den is None:
        raise ValueError(f"Sheet 'den' not found in {drillhole}_sample_report.xlsx")

    # Keep only the two columns we need; .copy() ensures df_out is an
    # independent frame, not a view into df_den (avoids SettingWithCopyWarning
    # on the assignments below)
    df_out = df_den[['sample_id', 'density']].copy()

    # Cast sample IDs to string up front — everything downstream (the 'cal'
    # substring check, merges with other prop tables) expects string IDs
    df_out['sample_id'] = df_out['sample_id'].astype(str)

    # Standardize column name to match the convention used across other
    # prop-processing functions (e.g. cond_sus_proc)
    df_out.rename(columns={'sample_id': 'sample id'}, inplace=True)

    # Calibration samples are tagged with 'cal' in their sample ID and are
    # reported separately from real measurements. na=False guards against
    # .str.contains raising on any unexpected NaN/non-string values
    cal_mask = df_out['sample id'].str.contains('cal', na=False)
    df_cal = df_out[cal_mask]

    # Real (non-calibration) samples, with a clean 0-based index
    df_final = df_out[~cal_mask].copy()
    df_final.index = range(len(df_final))

    # Insert placeholder rows for any sample IDs present in the master report
    # but absent from this measurement set, so row order/count matches
    # sample_id_report downstream
    df_final = find_missing_list_indices(df_final, 'sample id', sample_id_report)

    return df_final, df_cal

In [10]:
df, cal = density('TC-2531-001')

In [11]:
df.to_csv('check_density.csv')

# IP

In [13]:
def _load_ip_records(ip_res_dir):
    """
    Read every ip_records.csv across all date/instrument subfolders exactly
    once, concatenating into a single Record_Id -> measurement lookup table.
    """
    date_folders = []
    for name in os.listdir(ip_res_dir):
        try:
            date_folders.append((datetime.strptime(name, "%d-%m-%y"), name))
        except ValueError:
            continue  # not a date-named folder — skip it

    sorted_folder_names = [name for _, name in sorted(date_folders)]

    all_records = []
    for folder_name in sorted_folder_names:
        csv_path = ip_res_dir / folder_name / 'kt20_0385' / 'ip' / 'ip_records.csv'
        if csv_path.exists():
            all_records.append(pd.read_csv(csv_path))
        else:
            print(f"Warning: no ip_records.csv found for {folder_name}")

    if not all_records:
        return pd.DataFrame(columns=['Record_Id'])

    combined = pd.concat(all_records, ignore_index=True)

    dup_mask = combined['Record_Id'].duplicated()
    if dup_mask.any():
        dupes = combined.loc[dup_mask, 'Record_Id'].unique().tolist()
        print(f"Warning: duplicate Record_Id(s) across ip_records.csv files: {dupes}")
        combined = combined[~dup_mask]

    return combined


def ip_res(drillhole):
    ip_res_dir = Path(drillhole) / 'ip_res'

    data_inventory = pd.read_excel(f'{drillhole}/{drillhole}_sample_report.xlsx', sheet_name=None)
    df_record = data_inventory.get('ip_res')
    if df_record is None:
        raise ValueError(f"Sheet 'ip_res' not found in {drillhole}_sample_report.xlsx")

    df_record = df_record.reset_index(drop=True)

    ip_records = _load_ip_records(ip_res_dir)

    result_df = df_record[['sample_id', 'measurement_id']].merge(
        ip_records,
        left_on='measurement_id',
        right_on='Record_Id',
        how='left',
    )

    # Case 1: measurement_id was never recorded (NaN in the sample report)
    #   -> leave the row blank (real NaN) — no data was ever expected here.
    # Case 2: measurement_id WAS recorded, but no matching Record_Id was
    # found in any ip_records.csv -> mark the row 'missing', since data
    # should exist but couldn't be located.
    not_recorded = df_record['measurement_id'].isna().values   # .values forces positional
    no_match = result_df['Record_Id'].isna().values             # alignment, not index-based

    data_missing_mask = no_match & ~not_recorded

    result_df['sample_id'] = df_record['sample_id'].astype(str)
    result_df.rename(columns={'sample_id': 'sample id'}, inplace=True)
    result_df.drop(columns=['measurement_id', 'Record_Id'], inplace=True)

    # Case 1 rows: already blank/NaN from the left-merge — nothing to do.
    # Case 2 rows: overwrite every cell with 'missing', preserving sample id.
    result_df.loc[data_missing_mask, :] = 'missing'
    result_df.loc[data_missing_mask, 'sample id'] = df_record.loc[data_missing_mask, 'sample_id'].astype(str).values

    # Calibration samples are tagged with 'cal' in the sample ID and reported
    # separately from real measurements
    cal_mask = result_df['sample id'].str.contains('cal', na=False)
    df_cal = result_df[cal_mask]
    df_final = result_df[~cal_mask].copy()
    df_final.index = range(len(df_final))

    return df_final, df_cal

In [14]:
a,b =ip_res('TC-2531-001')

In [15]:
a.to_csv('check_ip.csv')

# Velocity

In [16]:
# Candidate (separator, header_row, sample_stride) configs to try, in order,
# for the different vendor export formats seen across folders. header_row and
# sample_stride are best-guess constants inherited from the original code —
# confirm these against the actual file layouts before relying on them.
READ_CONFIGS = [
    {'sep': ',', 'header': 14, 'stride': 25},
    {'sep': '\t', 'header': 10, 'stride': 16},
    {'sep': ';', 'header': 14, 'stride': 25},
]


def convert_csv_to_utf8(input_file):
    """
    Detect encoding and, if not already UTF-8, write a UTF-8 copy alongside
    the original. Returns the path that should actually be read downstream,
    so callers never have to re-detect encoding or guess which file to open.
    """
    with open(input_file, 'rb') as f:
        raw = f.read()
    enc = chardet.detect(raw)

    if enc['encoding'] and enc['encoding'].lower() == 'utf-8':
        print(f"The file {input_file} is already in UTF-8 encoding.")
        return input_file, 'utf-8'

    df = pd.read_csv(input_file, encoding=enc['encoding'])
    output_file = input_file.replace('.csv', '_utf8.csv')
    df.to_csv(output_file, encoding='utf-8', sep=',', index=False)
    print(f"File converted and saved as {output_file}")
    return output_file, 'utf-8'


def _read_vendor_csv(csv_path):
    """
    Try each known (sep, header, stride) combination until one produces an
    'ID' column. Returns (dataframe_of_sampled_rows) or None if nothing works
    — explicitly, so a failed parse can never fall through and reuse another
    folder's leftover data (the original's failure mode).
    """
    for cfg in READ_CONFIGS:
        try:
            df_data = pd.read_csv(csv_path, sep=cfg['sep'], encoding='utf-8', header=cfg['header'])
            df_data.columns = df_data.columns.str.strip()
            if 'ID' not in df_data.columns:
                continue
            sample_rows = list(range(1, len(df_data), cfg['stride']))
            return df_data.iloc[sample_rows]
        except (KeyError, pd.errors.ParserError):
            continue

    print(f"Warning: none of the known formats parsed {csv_path}")
    return None


def _load_vel_records(prop_dir):
    """
    Walk every date-named subfolder once, convert/read its raw vendor CSV,
    and concatenate the sampled rows into a single ID -> record lookup table.
    """
    date_folders = []
    for name in os.listdir(prop_dir):
        try:
            date_folders.append((datetime.strptime(name, "%d-%m-%y"), name))
        except ValueError:
            continue
    sorted_folder_names = [name for _, name in sorted(date_folders)]

    all_records = []
    for folder_name in sorted_folder_names:
        folder_path = prop_dir / folder_name

        raw_path = folder_path / 'Punditlink Data.csv'
        if not raw_path.exists():
            print(f"Warning: no 'Punditlink Data.csv' in {folder_path}")
            continue

        # Convert (if needed) and read the SAME file path back — removes the
        # original's disconnect between what was converted and what was read.
        utf8_path, _ = convert_csv_to_utf8(str(raw_path))
        sampled = _read_vendor_csv(utf8_path)
        if sampled is not None:
            all_records.append(sampled)

    if not all_records:
        return pd.DataFrame(columns=['ID'])

    combined = pd.concat(all_records, ignore_index=True)
    combined['ID'] = combined['ID'].astype(str).str.strip()

    # Guard against the same ID appearing more than once across folders —
    # the original had no way to detect this, which would silently duplicate
    # a row and shift everything after it out of position.
    dup_mask = combined['ID'].duplicated()
    if dup_mask.any():
        dupes = combined.loc[dup_mask, 'ID'].unique().tolist()
        print(f"Warning: duplicate ID(s) across vendor CSVs: {dupes}")
        combined = combined[~dup_mask]

    return combined


def vel(drillhole, prop):
    prop_dir = Path(drillhole) / prop

    data_inventory = pd.read_excel(f'{drillhole}/{drillhole}_sample_report.xlsx', sheet_name=None)
    df_record = data_inventory.get(prop)
    if df_record is None:
        raise ValueError(f"Sheet '{prop}' not found in {drillhole}_sample_report.xlsx")

    vel_records = _load_vel_records(prop_dir)

    # measurement_id may be float-typed (NaN-compatible) in df_record; build a
    # matching string key without mutating df_record itself
    measurement_id_str = df_record['measurement_id'].apply(
        lambda x: str(int(x)) if pd.notna(x) else pd.NA
    )

    # Left-merge preserves df_record's row order/count exactly and fills NaN
    # for any measurement_id with no vendor-file match — replaces the entire
    # manual "reinsert placeholder rows at tracked positions" block, which is
    # what let rows silently drift out of alignment in the original.
    result_df = pd.DataFrame({
        'sample id': df_record['sample_id'].astype(str),
        'measurement_id': measurement_id_str,
    }).merge(vel_records, left_on='measurement_id', right_on='ID', how='left')

    no_match = result_df['ID'].isna()
    result_df.loc[no_match & df_record['measurement_id'].isna(), 'status'] = 'missing'
    result_df.loc[no_match & df_record['measurement_id'].notna(), 'status'] = 'error'

    result_df.drop(columns=['measurement_id', 'ID'], inplace=True)
    result_df.index = range(len(result_df))

    return result_df

In [17]:
result_df = vel('G11-2531-01','v_p')

File converted and saved as G11-2531-01\v_p\06-04-23\Punditlink Data_utf8.csv


In [18]:
result_df.to_csv('check_vs.csv')

# XRF

In [19]:
import os
import glob
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime

# Single source of truth for the analyte columns used across all five
# groupby aggregations below — avoids five independently-maintained copies
# of the same 37-element list drifting out of sync if elements are added/removed
ELEMENT_COLS = [
    'Mg Concentration', 'Al Concentration', 'Si Concentration', 'P Concentration',
    'S Concentration', 'K Concentration', 'Ca Concentration', 'Ti Concentration',
    'V Concentration', 'Cr Concentration', 'Mn Concentration', 'Fe Concentration',
    'Co Concentration', 'Ni Concentration', 'Cu Concentration', 'Zn Concentration',
    'As Concentration', 'Se Concentration', 'Rb Concentration', 'Sr Concentration',
    'Y Concentration', 'Zr Concentration', 'Nb Concentration', 'Mo Concentration',
    'Ag Concentration', 'Cd Concentration', 'Sn Concentration', 'Sb Concentration',
    'W Concentration', 'Au Concentration', 'Hg Concentration', 'Pb Concentration',
    'Bi Concentration', 'Th Concentration', 'U Concentration', 'LE Concentration',
]

# (sep, header) combinations to try in order for the different XRF export formats
READ_CONFIGS = [
    {'sep': ',', 'header': 1},
    {'sep': ',', 'header': 0},
    {'sep': ';', 'header': 0},
    {'sep': '\t', 'header': 0},
]


def extract_last_four_digits(number):
    return float(str(number)[-4:])


def _read_xrf_csv(csv_path):
    """
    Try each known format until one produces an 'info' column.
    Returns None (rather than reusing a stale DataFrame) if nothing works,
    so a failed folder contributes zero rows instead of silently duplicating
    the previous folder's data — the original's fallthrough bug.
    """
    for cfg in READ_CONFIGS:
        try:
            df_xrf = pd.read_csv(csv_path, sep=cfg['sep'], skiprows=1 if cfg['header'] == 1 else 0)
            if 'info' in df_xrf.columns:
                return df_xrf
        except (KeyError, pd.errors.ParserError):
            continue
    print(f"Warning: cannot read {csv_path} with any known format")
    return None


def _load_xrf_records(xrf_dir):
    date_folders = []
    for name in os.listdir(xrf_dir):
        try:
            date_folders.append((datetime.strptime(name, "%d-%m-%y"), name))
        except ValueError:
            continue
    sorted_folder_names = [name for _, name in sorted(date_folders)]

    all_records = []
    for folder_name in sorted_folder_names:
        folder_path = xrf_dir / folder_name
        csv_files = glob.glob(str(folder_path / '*.csv'))
        if not csv_files:
            print(f"Warning: no CSV in {folder_path}")
            continue
        df_xrf = _read_xrf_csv(csv_files[-1])
        if df_xrf is not None:
            all_records.append(df_xrf)

    return pd.concat(all_records, ignore_index=True) if all_records else pd.DataFrame()


def xrf(drillhole):
    xrf_dir = Path(drillhole) / 'xrf'

    data_inventory = pd.read_excel(f'{drillhole}/{drillhole}_sample_report.xlsx', sheet_name=None)
    df_record = data_inventory.get('sample_report')
    sample_id_report = df_record['sample_id'].to_list()

    df_proc = _load_xrf_records(xrf_dir)
    info = df_proc['info'].tolist()

    # Rows tagged as calibration/standard/blank material — excluded from
    # the real-sample table entirely
    drop_list = [
        i for i, val in enumerate(info)
        if isinstance(val, str) and (val.startswith('cal') or val.startswith('std') or val.startswith('blk'))
    ]

    # Standards specifically (confirmed: literal 'std' prefix) are pulled out
    # separately into new_cal for reporting alongside the main result
    cal_list = [i for i, val in enumerate(info) if isinstance(val, str) and val.startswith('std')]

    new_df = df_proc.drop(df_proc.index[drop_list])
    new_cal = df_proc.iloc[cal_list].reset_index(drop=True)

    # Renumber standards sequentially (cal_0, cal_1, ...) rather than keeping
    # their original info strings
    new_cal['info'] = ['cal_0' + str(i) for i in range(len(new_cal))]
    new_cal = new_cal[['info', 'As Concentration', 'As Error1s']].rename(columns={'info': 'sample id'})

    new_df = new_df.drop(new_df[new_df['info'] == 'del'].index)

    # Sample IDs in the raw 'info' field carry a trailing letter suffix
    # (e.g. replicate/position marker) that isn't part of the true sample ID
    new_df['info'] = new_df['info'].astype(str).str[:-1]
    new_df.index = range(len(new_df))

    # Drop any row whose (suffix-stripped) info isn't a known sample ID at all
    str_sample_id_report = [str(x) for x in sample_id_report]
    keep_mask = new_df['info'].isin(str_sample_id_report)
    new_df = new_df[keep_mask].reset_index(drop=True)

    # Column cleanup: drop instrument metadata, error columns, and 'User' columns
    new_df = new_df.dropna(axis=1, how='all')
    new_df.columns = new_df.columns.str.strip()
    new_df = new_df.drop(
        columns=['Instrument Serial Num', 'Reading #', 'Date', 'Time',
                 'Method Name', 'Test Label', 'Collimation Status', 'Units'],
        errors='ignore',
    )
    new_df = new_df.drop(columns=new_df.filter(regex='Error').columns)
    new_df = new_df.drop(columns=new_df.filter(regex='User').columns)

    # Below-detection-limit readings are set to zero here — note for the
    # methods section that this will understate true concentration relative
    # to e.g. an LOD/2 substitution, particularly for trace elements with
    # many <LOD readings
    new_df = new_df.replace('<LOD', '0.0')
    new_df = new_df.replace({'na': np.nan, 'de': np.nan})

    # Numeric columns via pd.to_numeric(errors='coerce') rather than a hard
    # .astype(float) — an unexpected token becomes NaN instead of crashing
    # the whole pipeline on one bad cell
    info_col = new_df['info']
    numeric_df = new_df.drop(columns=['info']).apply(pd.to_numeric, errors='coerce')
    new_df = pd.concat([numeric_df, info_col], axis=1)
    new_df['temp_info'] = new_df['info'].apply(extract_last_four_digits)

    # temp_info (last 4 digits) is confirmed unique per sample, so it's a
    # safe groupby/join key — build an explicit temp_info -> sample_id
    # lookup now, rather than relying on np.unique(sampleid_list) lining up
    # positionally with the groupby's own (numeric) sort order, which is
    # what could previously mislabel rows whenever the two orderings diverged
    id_lookup = new_df[['temp_info', 'info']].drop_duplicates(subset='temp_info')

    agg_funcs = {'Mean': 'mean', 'Std': 'std', 'Median': 'median', 'Max': 'max', 'Min': 'min'}
    agg_frames = []
    for suffix, func in agg_funcs.items():
        grouped = new_df.groupby('temp_info')[ELEMENT_COLS].agg(func).reset_index()
        grouped.columns = ['temp_info'] + [col.replace('Concentration', suffix) for col in ELEMENT_COLS]
        agg_frames.append(grouped)

    # Merge all five stats frames on temp_info (safe: same groupby, same key,
    # same row order guaranteed by construction) rather than a positional concat
    result_df = agg_frames[0]
    for frame in agg_frames[1:]:
        result_df = result_df.merge(frame, on='temp_info')

    # Attach true sample IDs via explicit merge on temp_info instead of a
    # positionally-assumed np.unique(sampleid_list) assignment
    result_df = result_df.merge(id_lookup, on='temp_info', how='left')
    result_df = result_df.drop(columns=['temp_info']).rename(columns={'info': 'sample id'})

    # --- Align result_df to sample_id_report, marking unmatched rows ---
    #
    # reindex against sample_id_report's own last-four-digit keys guarantees
    # output row order/count exactly matches sample_id_report, with any
    # unmatched sample id landing at its correct position automatically —
    # no manual position-tracking/concat-at-index needed.
    report_keys = [int(str(x)[-4:]) for x in sample_id_report]

    # Sanity check: reindex silently duplicates a row into every position
    # sharing a key, so a non-unique key set would corrupt row alignment
    # rather than error out. Confirm before trusting this path.
    if pd.Series(report_keys).duplicated().any():
        dupe_keys = pd.Series(report_keys)[pd.Series(report_keys).duplicated()].unique().tolist()
        raise ValueError(
            f"sample_id_report has non-unique last-four-digit keys: {dupe_keys} "
            "— reindex-based alignment is not safe with duplicate keys."
        )

    result_df['_match_key'] = result_df['sample id'].apply(lambda x: int(str(x)[-4:]))
    result_df = result_df.set_index('_match_key').reindex(report_keys)

    missing_mask = result_df.isna().all(axis=1)
    if missing_mask.any():
        missing_ids = [sid for sid, is_missing in zip(sample_id_report, missing_mask) if is_missing]
        print(f"Warning: no XRF match for {len(missing_ids)} sample id(s): {missing_ids}")

    # Fill every column with 'missing' for unmatched rows, then restore the
    # correct sample id string in those rows
    result_df.loc[missing_mask, :] = 'missing'
    result_df.loc[missing_mask, 'sample id'] = [
        str(s) for s, m in zip(sample_id_report, missing_mask) if m
    ]

    result_df = result_df.reset_index(drop=True)

    return result_df, new_cal

In [20]:
df, cal = xrf('TC-2531-001')

In [21]:
df.to_csv('check_xrf.csv')